# FailCatcher — Quick-start Tutorial

**FailCatcher** is a post-hoc uncertainty quantification (UQ) library for PyTorch models.  
This notebook shows the full pipeline on **CIFAR-10** in ~5 minutes:

1. Install the library
2. Download a pretrained ResNet-18 from HuggingFace
3. Run inference on the CIFAR-10 test set
4. Compute **MSR** (Maximum Softmax Response) uncertainty scores
5. Evaluate with **AUROC-f**, **AURC**, and **AUGRC**
6. Visualise the most uncertain samples
7. Save evaluation plots

> **Install**: `pip install FailCatcher`  
> **Paper**: [Steinmetz et al., medRxiv 2026](https://www.medrxiv.org/content/10.64898/2026.05.04.26350496v1)
> **Repo**: [pstnmz/FailCatcher](https://github.com/pstnmz/FailCatcher)


## Step 0 — Install dependencies


In [ ]:
!pip install -q FailCatcher timm torchvision huggingface_hub matplotlib


## Step 1 — Imports


In [ ]:
import torch
import numpy as np
import timm
import torch.nn as nn
import matplotlib.pyplot as plt
from torchvision.datasets import CIFAR10
from torchvision import transforms
from torch.utils.data import DataLoader
from huggingface_hub import hf_hub_download


## Step 2 — Download a pretrained model from HuggingFace

We load [`edadaltocg/resnet18_cifar10`](https://huggingface.co/edadaltocg/resnet18_cifar10):  
a ResNet-18 fine-tuned on CIFAR-10 (~94 % accuracy).  
The CIFAR-10 variant uses a **3×3 stem** and **no max-pool**, so we patch the architecture before loading weights.


In [ ]:
print('Downloading ResNet18-CIFAR10 from HuggingFace...')
ckpt_path = hf_hub_download(repo_id='edadaltocg/resnet18_cifar10',
                             filename='pytorch_model.bin')

# Standard ResNet18 modified for CIFAR-10: 3x3 stem, no maxpool
model = timm.create_model('resnet18', pretrained=False, num_classes=10)
model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
model.maxpool = nn.Identity()
model.load_state_dict(torch.load(ckpt_path, map_location='cpu'))
model.eval()
print('  -> Model loaded')


## Step 3 — CIFAR-10 test dataset

10 000 test images across 10 classes: airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck.


In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4914, 0.4822, 0.4465],
                         std=[0.2470, 0.2435, 0.2616]),
])

test_dataset = CIFAR10(root='./data', train=False, download=True, transform=transform)
test_loader  = DataLoader(test_dataset, batch_size=256, shuffle=False, num_workers=4)
print(f'Test set: {len(test_dataset)} samples, 10 classes')


## Step 4 — Run inference

`evaluate_models_on_loader` runs the model on the full test set and returns:
- `y_true`: ground-truth labels
- `y_scores`: softmax probabilities `[N, C]`
- `preds`: argmax predictions
- `correct_idx` / `incorrect_idx`: indices of correct / incorrect samples


In [ ]:
from UQ_Toolbox.core.utils import evaluate_models_on_loader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
model = model.to(device)

y_true, y_scores, preds, correct_idx, incorrect_idx, _ = \
    evaluate_models_on_loader([model], test_loader, device)

print(f'Accuracy: {len(correct_idx)/len(y_true):.3f}  '
      f'| Correct: {len(correct_idx)}  Incorrect: {len(incorrect_idx)}')


## Step 5 — Compute uncertainty scores (MSR)

**Maximum Softmax Response (MSR)** is the simplest post-hoc UQ method:

$$\text{uncertainty}(x) = 1 - \max_c \, p(y=c \mid x)$$

A high MSR score means the model's top class has low confidence — likely a failure.


In [ ]:
from UQ_Toolbox.methods.distance import DistanceToHardLabelsMethod

uncertainties = DistanceToHardLabelsMethod().compute(y_scores)
print(f'Uncertainty scores — min: {uncertainties.min():.4f}  '
      f'max: {uncertainties.max():.4f}  mean: {uncertainties.mean():.4f}')


## Step 6 — Evaluate: AUROC-f, AURC, AUGRC

| Metric | Interpretation | Direction |
|--------|---------------|-----------|
| **AUROC-f** | How well uncertainty ranks failures above correct predictions | ↑ higher is better |
| **AURC** | Area under the selective-risk / coverage curve | ↓ lower is better |
| **AUGRC** | Average rate of silent failures across all coverage thresholds (Traub et al., NeurIPS 2024) | ↓ lower is better |


In [ ]:
from UQ_Toolbox.evaluation.evaluation import compute_all_metrics

metrics = compute_all_metrics(uncertainties, preds, y_true, correct_idx, incorrect_idx)

print('── Results ──────────────────────────────')
for k, v in metrics.items():
    print(f'  {k:<12}: {v:.4f}')


## Step 7 — Visualise the most uncertain samples

The 10 samples with the highest MSR score.  
Titles are **red** when misclassified, **green** when correct.  
At ~94 % accuracy, most of these will be wrong predictions — exactly what a good UQ method should flag.


In [ ]:
import os
os.makedirs('./figures', exist_ok=True)

classes = ['airplane', 'automobile', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck']

top10_idx = np.argsort(uncertainties)[-10:][::-1]  # highest uncertainty first

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
fig.suptitle('Top 10 Most Uncertain Samples (MSR)', fontsize=14, fontweight='bold')

for ax, idx in zip(axes.flat, top10_idx):
    img      = test_dataset.data[idx]   # raw uint8 [32,32,3]
    true_cls = classes[y_true[idx]]
    pred_cls = classes[preds[idx]]
    unc      = uncertainties[idx]
    correct  = idx in set(correct_idx)

    ax.imshow(img)
    ax.set_title(f'unc={unc:.3f}\ntrue: {true_cls}\npred: {pred_cls}',
                 fontsize=8, color='green' if correct else 'red')
    ax.axis('off')

plt.tight_layout()
fig.savefig('./figures/top10_uncertain.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: ./figures/top10_uncertain.png')


## Step 8 — Save evaluation plots

Generates three figures in `./figures/`:
- `MSR_roc_curve.png` — ROC curve for failure prediction (AUROC-f)
- `MSR_risk_coverage.png` — Risk-coverage and generalized-risk curves (AURC / AUGRC)
- `MSR_distributions.png` — Uncertainty distributions for correct vs incorrect predictions


In [ ]:
from UQ_Toolbox.evaluation.evaluation import save_all_evaluation_plots

paths = save_all_evaluation_plots(
    uncertainties, preds, y_true,
    method_name='MSR',
    output_dir='./figures',
    correct_idx=correct_idx,
    incorrect_idx=incorrect_idx,
)
print('── Figures saved ────────────────────────────')
for name, path in paths.items():
    print(f'  {name:<16}: {path}')
